# HTML Video

Browse bundled HTML video templates, inspect a template, and render configured WebM frames or media ports through the app MCP tools.

In [ ]:
declare const spur: any;
const encoder = new TextEncoder();
const decoder = new TextDecoder();

async function readExactly(reader: { read(p: Uint8Array): Promise<number | null> }, size: number): Promise<Uint8Array> {
  const buffer = new Uint8Array(size);
  let offset = 0;
  while (offset < size) {
    const n = await reader.read(buffer.subarray(offset));
    if (n === null) throw new Error("notebook MCP socket closed");
    offset += n;
  }
  return buffer;
}

async function readFrame(conn: Deno.Conn): Promise<any> {
  const header = await readExactly(conn, 4);
  const length = new DataView(header.buffer, header.byteOffset, header.byteLength).getUint32(0, false);
  return JSON.parse(decoder.decode(await readExactly(conn, length)));
}

async function writeFrame(conn: Deno.Conn, value: unknown): Promise<void> {
  const payload = encoder.encode(JSON.stringify(value));
  const frame = new Uint8Array(4 + payload.length);
  new DataView(frame.buffer).setUint32(0, payload.length, false);
  frame.set(payload, 4);
  await conn.write(frame);
}

async function callNotebookTool(name: string, args: Record<string, unknown>): Promise<any> {
  const socketPath = Deno.env.get("SPUR_NOTEBOOK_MCP_SOCKET");
  if (!socketPath) throw new Error("SPUR_NOTEBOOK_MCP_SOCKET is not set");
  const conn = await Deno.connect({ transport: "unix", path: socketPath });
  let id = 1;
  try {
    await writeFrame(conn, {
      jsonrpc: "2.0",
      id: id++,
      method: "initialize",
      params: {
        protocolVersion: "2025-11-25",
        capabilities: {},
        clientInfo: { name: "html-video-app", version: "0" }
      }
    });
    await readFrame(conn);
    await writeFrame(conn, { jsonrpc: "2.0", method: "notifications/initialized", params: {} });
    const requestId = id++;
    await writeFrame(conn, { jsonrpc: "2.0", id: requestId, method: "tools/call", params: { name, arguments: args } });
    const response = await readFrame(conn);
    if (response.error) throw new Error(response.error.message ?? JSON.stringify(response.error));
    const result = response.result ?? {};
    if (result.structuredContent) return result.structuredContent;
    if (result.structured_content) return result.structured_content;
    const text = result.content?.find?.((item: any) => item.type === "text")?.text;
    if (typeof text === "string") {
      try { return JSON.parse(text); } catch (_) { return { text }; }
    }
    return result;
  } finally {
    try { conn.close(); } catch (_) {}
  }
}

(globalThis as any).htmlVideoCallTool = callNotebookTool;
const searchResult = await callNotebookTool("html_video_search_templates", {
  intent: "product launch motion typography dashboard intro",
  top: 8
});
const items = Array.isArray(searchResult.items) ? searchResult.items : [];
(globalThis as any).htmlVideoTemplateResults = items;
(globalThis as any).htmlVideoSelectedTemplateId = items[0]?.id ?? "basic";

const anywidget = await spur.anywidget();
anywidget.widget({
  state: { items, selectedId: (globalThis as any).htmlVideoSelectedTemplateId },
  render({ model, el }: any) {
    const items = model.get("items") ?? [];
    const selectedId = model.get("selectedId");
    el.innerHTML = `
      <style>
        .hv-search{padding:24px;font:14px system-ui,sans-serif;color:#172033;background:#fbfcfe}
        .hv-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(220px,1fr));gap:12px;margin-top:16px}
        .hv-card{border:1px solid #d9e2ec;background:white;border-radius:8px;padding:14px;box-shadow:0 1px 2px rgba(15,23,42,.04)}
        .hv-card[data-selected="true"]{border-color:#2563eb;box-shadow:0 0 0 2px rgba(37,99,235,.16)}
        .hv-title{font-weight:700;margin-bottom:6px}.hv-tags{display:flex;gap:6px;flex-wrap:wrap;margin-top:10px}.hv-tag{font-size:11px;border:1px solid #dbe3ef;border-radius:999px;padding:2px 7px;background:#f8fafc}
      </style>
      <section class="hv-search">
        <h2>Template Browser</h2>
        <div class="hv-grid">
          ${items.map((item: any) => `<article class="hv-card" data-selected="${item.id === selectedId}"><div class="hv-title">${item.title ?? item.id}</div><div>${item.summary ?? item.intent ?? ""}</div><div class="hv-tags">${(item.tags ?? []).map((tag: any) => `<span class="hv-tag">${tag}</span>`).join("")}</div></article>`).join("")}
        </div>
      </section>`;
  }
})

In [ ]:
declare const spur: any;
const callTool = (globalThis as any).htmlVideoCallTool;
if (typeof callTool !== "function") throw new Error("template search cell has not initialized the MCP client");
const templateId = (globalThis as any).htmlVideoSelectedTemplateId ?? "basic";
const template = await callTool("html_video_get_template", { id: templateId });
(globalThis as any).htmlVideoTemplate = template;

const anywidget = await spur.anywidget();
anywidget.widget({
  state: {
    title: template.metadata?.title ?? template.id,
    id: template.id,
    summary: template.metadata?.summary ?? "",
    html: template.html ?? "",
    skill: template.skill_md ?? ""
  },
  render({ model, el }: any) {
    const html = model.get("html") ?? "";
    el.innerHTML = `
      <style>
        .hv-preview{display:grid;grid-template-columns:minmax(280px,1fr) minmax(280px,1.2fr);gap:18px;padding:24px;font:14px system-ui,sans-serif;color:#172033;background:white}
        .hv-pane{min-height:220px;border:1px solid #d9e2ec;border-radius:8px;background:#fbfcfe;overflow:hidden}.hv-meta{padding:18px}.hv-meta h2{margin:0 0 8px}.hv-code{margin-top:14px;max-height:180px;overflow:auto;border:1px solid #e2e8f0;background:#0f172a;color:#e2e8f0;border-radius:6px;padding:12px;font:12px ui-monospace,monospace;white-space:pre-wrap}.hv-frame{width:100%;height:100%;min-height:360px;border:0;background:white}
      </style>
      <section class="hv-preview">
        <div class="hv-pane hv-meta"><h2>${model.get("title")}</h2><div>${model.get("summary")}</div><pre class="hv-code">${html.replace(/[<&]/g, (c: string) => c === "<" ? "&lt;" : "&amp;")}</pre></div>
        <div class="hv-pane"><iframe class="hv-frame" sandbox="allow-scripts" srcdoc="${html.replace(/&/g, "&amp;").replace(/"/g, "&quot;")}"></iframe></div>
      </section>`;
  }
})

In [ ]:
declare const spur: any;
const callTool = (globalThis as any).htmlVideoCallTool;
if (typeof callTool !== "function") throw new Error("template search cell has not initialized the MCP client");
const defaultOutput = `${Deno.cwd()}/html-video-render.mp4`;
const request = (globalThis as any).htmlVideoRenderRequest ?? null;
let renderResult = null;
let renderError = null;
if (request) {
  try {
    renderResult = await callTool("html_video_render", request);
  } catch (error) {
    renderError = error instanceof Error ? error.message : String(error);
  }
}

const anywidget = await spur.anywidget();
anywidget.widget({
  state: { request, renderResult, renderError, defaultOutput },
  render({ model, el }: any) {
    const request = model.get("request");
    const result = model.get("renderResult");
    const error = model.get("renderError");
    el.innerHTML = `
      <style>
        .hv-render{padding:24px;font:14px system-ui,sans-serif;color:#172033;background:#f8fafc}.hv-panel{max-width:760px;border:1px solid #d9e2ec;border-radius:8px;background:white;padding:18px}.hv-row{display:grid;grid-template-columns:150px 1fr;gap:10px;align-items:center;margin-top:10px}.hv-input{border:1px solid #cbd5e1;border-radius:6px;padding:8px;background:#f8fafc;font:13px ui-monospace,monospace}.hv-status{margin-top:16px;border-radius:6px;padding:12px;background:#eef6ff;border:1px solid #bfdbfe}.hv-error{background:#fff1f2;border-color:#fecdd3}.hv-json{white-space:pre-wrap;font:12px ui-monospace,monospace}</style>
      <section class="hv-render"><div class="hv-panel"><h2>Render Controls</h2><div class="hv-row"><label>Output path</label><div class="hv-input">${request?.output_path ?? model.get("defaultOutput")}</div></div><div class="hv-row"><label>Resolution</label><div class="hv-input">${request?.resolution ?? "1280x720"}</div></div><div class="hv-row"><label>FPS</label><div class="hv-input">${request?.fps ?? 30}</div></div>${result ? `<div class="hv-status"><strong>Rendered</strong><pre class="hv-json">${JSON.stringify(result, null, 2)}</pre></div>` : error ? `<div class="hv-status hv-error"><strong>Render error</strong><pre class="hv-json">${error}</pre></div>` : `<div class="hv-status"><strong>Ready</strong><div>Render request is waiting for webm_frames or port_names.</div></div>`}</div></section>`;
  }
})